# 航班清洗与同机衔接

本 notebook 说明五个月 BTS 原始航班如何转成可审计的清洗表与连续航段。完整处理逻辑只维护在 `scripts/prepare_flights.py`；notebook 负责讲解并调用同一份代码。

This notebook explains the five-month BTS cleaning and same-aircraft linking workflow. The script is the single source of processing logic.


## 1. 输入与目录

从包根目录或 `notebooks/` 目录运行都可以。输入为 `raw/flights/2026_04.zip` 至 `2026_08.zip`，以及 `references/airports.csv`、`references/airport_codes.csv`。后两份机场表分别提供稳定机场实体与机场代码有效日期；机场代码按航班日期匹配 ID，因此 PBI/DJT 可作为同一机场实体参与连续性检查。

Run from the package root or this notebook directory. Airport codes are resolved by flight date to stable airport IDs.


## 2. 规则与边界

- 保留取消、备降、缺尾号及缺失/异常时刻记录，先依尾号和计划出发 UTC 排序建立相邻序列；它们不会被跳过来制造虚假的前后航段。
- 使用出发机场当地日期和 HHMM 构造计划 UTC；`2400` 表示次日 00:00。DST 歧义/不存在的本地时间视为无效。
- 按原逻辑核对计划到达钟表、实际出发/到达钟表、实际历时及 BTS 延误字段，容差为 1 分钟。
- 负的 `DepDelay`/`ArrDelay` 原值保留；`DepDelayMinutes`/`ArrDelayMinutes` 是非负分钟口径。衔接变化 `delta_delay_minutes` = 下游非负出发延误分钟 − 上游非负到达延误分钟。
- 有效连接要求同一稳定机场 ID 连续、双段正常且时间有效、顺序无冲突、计划周转为 0–1440 分钟、实际地面时间非负。跨月相邻航班照常保留。预测时点是上游实际到达。

Cancellations, diversions, missing tails and invalid times remain present while the sequence is constructed. Delay values and nonnegative delay minutes are both retained. The prediction timestamp is the upstream actual arrival.


## 3. 运行与输出

使用项目环境：`python`。在项目根目录打开终端执行：

```bash
python scripts/prepare_flights.py
```

仅运行小型合成边界断言（不读航班数据）：`python scripts/prepare_flights.py --check`。Notebook 下方单元调用同一脚本执行完整清洗。输出写入 `processed/flights_clean.csv.gz`、`processed/connection_candidates.csv.gz`、`processed/flight_connections.csv.gz`，以及 `tables/cleaning_summary.json`、`tables/connection_exclusions.csv`。Candidates 包含首个排除原因和逐项检查结果；accepted 表只含全部规则通过的连接。


In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
package_root = next((p for p in candidates if (p / 'scripts' / 'prepare_flights.py').is_file()), None)
if package_root is None and cwd.name == 'notebooks' and (cwd.parent / 'scripts' / 'prepare_flights.py').is_file():
    package_root = cwd.parent
assert package_root is not None, 'Run from package root or notebooks/.'
sys.path.insert(0, str(package_root / 'scripts'))
from prepare_flights import run

summary = run()
summary


## 4. 执行边界

本次交付只做脚本语法和 notebook 结构检查；没有运行五个月清洗、模型或性能测试。首次执行会读取全部五个压缩 CSV，并生成上述文件。

This handoff has only static syntax and notebook-structure checks. The full five-month run, modeling and performance testing were not run.
